### Abschlussbericht: Erkenntnisse, Kritische Reflexion & Ausblick

#### Erkenntniszusammenfassung 

Die Untersuchugn mit verschiedenen Modellansätze zeigen ein hohe Korrelation der (logarithmierte) Like-Anzahl (`likes_log`) von Instagram-Posts im Bezug auf Metatdaten wie die Menge an Followern und Kommentaren als auch bildliche Merkmale. Letzteres zeigte in unserem Fall keine hohe Gewichtung.

#### Überblick der Modellleistungen

| Modell                        | Datenquelle         | Test-MSE  | Test-R²  | Kurze Einordnung                                           |
|:------------------------------|:--------------------|:----------|:---------|:------------------------------------------------------------|
| **OLS-Regression**            | Metadaten           | ≈ 4.00    | ≈ 0.65   | Solide Baseline, aber lineare Annahmen limitieren Korrelationen. |
| **Random Forest (RF)**        | Metadaten           | ≈ 1.50    | ≈ 0.87   | Sehr gute Leistung, fängt nichtlineare Effekte und Interaktionen ein. |
| **CNN (Bild-Mittelwert)**     | Bild-Features       | ≈ 15.87   | ≈ –0.26  | Geringe Vorhersagekraft – Datenbasis zu klein, Aggregation zu einfach. |
| **Stacking (RF + CNN)**       | Metadaten + Bild    | ≈ 1.60    | ≈ 0.873  | Minimal besser als RF; Bildsignal wird kaum gewichtet.       |

Das Random Forest Modell liefert die stärkste Vorhersage (R² ≈ 0.87) und erklären damit praktisch den größten Teil der Varianz. Die drei wichtigsten Prädikatoren in den Metadaten stellen dabei die Anzahl der Kommentare (stärkster Einfluss), die Anzahl der Follower und zuletzt Accounts von Musikern. Negativer Einfluss wurde bei der Anzahl von Hashtags festgestellt.
Das Extrahieren von bildlichen Merkmalen durch das vortrainiertes Modell ResNet50 brachte kaum Mehrwert (R² gering und teils negativ). 
Die Zusammenführung von Metadaten und Bildmerkmalen verbessert sich nur marginal gegenüber RF allein (R² ≈ 0.873 -> R² ≈ 0.8695).

#### Kontext und Relevanz
  - Eine valide Like-Prognose auf Basis von Metadaten allein ist in vielen beruflichen Anwendungen ausreichend (Agenturen, Brand Manager, Influencer).  
  - Vor allem Follower- und Kommentar-Zahlen sind leicht zugänglich und korrelieren stark mit dem Engagement.  
  - Aktuelle Erscheinung der Ergebnisse: Ein einfaches RF-Modell kann in einem Tool integriert werden, um Content Creator:innen zu zeigen, wie viele Likes sie realistischerweise erwarten können – bevor der Beitrag online geht. Aber:

#### Kritische Reflexion -> Limitationen und potenzielle Fehlerquellen
- Stichprobengröße (1968 Posts, hauptsächlich „Digital creator“) begrenzt Repräsentativität (Generalisierbarkeit) und steigert Homogenisierung und erzeugt negative CNN-Vorhersagen. Große, heterogene Instagram-Datensätze würden Bildmodelle deutlich aufwerten.
- Merkmale wie die Account-Kategorie oder Hashtags können je nach Nische (Essen/Mode/Fitness) stark variieren und als eigenes Modell nach Kategorie möglicherweise validere Ergebnisse bringen.
- Residuen plots zeigen starke Abweichungen, insbesondere bei sehr niedrigen oder sehr hohen Like-Zahlen. Selbst ohne strikte Annahmen wie Lineari­tät (OSL) im RF geben Feature-Importances nur globales Ranking, keine lokalen Zusammenhänge und mindern die Interpretierbarkeit. Partial-Dependence-Plots oder SHAP-Werte könnten helfen, lokales Verhalten besser zu verstehen.
- Zusätzliche Metadaten-Merkmale (z. B. Caption-Sentiment, Uhrzeit-Embeddings) wurden nicht implementiert, könnten aber weitere Varianz erklären.
- Accounts mit vielen Followern und hohem Engagement sind stärker vertreten, kleinere Nischen-Accounts (unter 10 000 Follower) kaum. Das Modell könnte systematisch überschätzen, wie viele Likes ein kleiner Account erhält, wenn die Trainingsdaten überwiegend große Accounts abdecken.
- Eye-Tracking-Analysen zeigen, dass der visuelle Eindruck maßgeblich darüber entscheidet, ob Nutzer die Caption näher betrachtet oder kommentiert. Weitere Analysen zeigen auch, dass kurze Captions (< 30 Wörter) deutlich stärkeres Engagement der Nutzer erzielen und Bildunterschriften häufig ganz ignoriert oder nicht vollständig gelesen werden. So sollte auch statt dem arithmetischen Mittel, die Gewichtung des ersten Bildes im Carousell, höhere Bedeutung zukommen.  
https://www.socialinsider.io/blog/instagram-caption-length/
https://www.researchgate.net/publication/352003748_Show_products_or_show_people_an_eye-tracking_study_of_visual_branding_strategy_on_Instagram 
- CNN und Stack-Limitationen:
  - Es können kaum robuste Patterns extrahiert werden, aufgrund des kleinen Datensatzes. CNN-Datenmenge unzureichend für robuste visuelle Feature-Learning
  - Zudem ist ResNet50 vorkonditioniert auf ImageNet-Klassen (Alltagsobjekte) also Objekterkennung und fängt keine spezielle Social-Media-Ästhetik ein (Filter, Textüberlagerungen). Verschiebung der lokalen Datenuntersuchung für Fine-Tuning auf Instagram-spezifischen Stilen könnte die Bildleistung verbessern. Hier ist jedoch Vorsicht mit Datenschutz geboten.
  - Bildaggregationsmethode (arithmetischer Mittelwert über alle Carousel-Bilder) vernachlässigt, dass die Bildreihenfolge und dass oft nur das erste Bild das Engagement bestimmt.
  - Unser Einfachmodell ignoriert gesondert Bildkomposition, Farbpalette, Textüberlagerung etc., sodass neben reinen Pixel-Features z. B. Farbanalysen, Objekterkennung (Gesichter, Markenlogos) oder Ästhetikmetriken wertvolle Informationen liefern. 
  - Den Metaregressor erwietern und anstelle einer rein linearen Regression Kreuzvalidierung auf Meta-Ebene Overfitting zwischen Einzel- und Meta-Level vermeiden. 
  - Geringe Gewichtung CNN: Lineare Meta-Regression unterschätzt nichtlineare Interaktionen 
- Kontinuierliche Modellüberwachung (regelmäßiges Retraining mit neuen Daten) ist essenziell, um gute Annahmen trotz regelmäsßiger Instagram-Algorithmen Änderungen (Feed-Ranking, Reels-Vorzug) zu gewährleisten. In der Praxis sind Posts also oft zeitabhängig: Trends, Saisonaleffekte (Feiertage, Events), die zu Leaks im Modell führen könnten, sodass zukünftig kein 60/40-Split genutzt werden sollte, ohne zeitliche Reihenfolge zu beachten.
- Intergrierung von Videosegmenten: Validierung der Vorhersage durch Anbindung eines Tools zur Verarbeitung und Analyse von Videodateien, da sie auch einen großen Teil des Instagram-Contents ausmachen.
- Für reale Anwendungen empfiehlt es sich, mit RF zu starten, kontinuierlich neue Daten zu sammeln, Modell-Drift zu überwachen und Bild-Modelle erst hinzuziehen, wenn ein ausreichend großer, qualitativ hochwertiger Bilddatensatz vorhanden ist, sodass letzendlich auch das Stack-Modell den Vorhersagewert erhöht.
- Rechenressource sollte für höhrere Datenmengen und Training auf mehr Epochen angepasst und sichergestellt werden


#### Ausblick

##### Web-Interface für Like-Prognosen
Eine benutzerfreundliche Weboberfläche mit:
- Upload-Funktion: Nutzer können ihre Bilder (und Videos) hochladen und relevante Metadaten definieren (Account-Kategorie, Follower-Anzahl, Hashtag-Strategie)
- KI-gestützte Analyse: Integration des Modells zur Echtzeit-Prognose
- Vergleichsvisualisierung: Verschiedene Posting-Strategien können gegenübergestellt werden
- Empfehlungssystem: Automatische Vorschläge zur Optimierung des Engagements
- Datenbasis: Kontinuierliche Modellanreicherung mit neuen Trainingsdaten für präzisere Vorhersagen
- API-Schnittstelle: Mögliche Integration in Content-Management-Systeme und Social-Media-Planungstools.

Die Weboberfläche würde Content-Erstellern ermöglichen, datenbasierte Entscheidungen zu treffen und ihre Social-Media-Strategie zu optimieren, bevor Inhalte überhaupt veröffentlicht werden. Damit kann jeder Post zielbringender gestaltet und zeitlich platziert werden. Social Media Platformen wie Instagram sind eine der Hauptwerbeflächen für Unternehmen, wozu oft Influencer:innen herangezogen werden, da sie beruflich mit ihrem Content überzeugen. Dieses Web-Interface stellt geeignetes Tool für die Unterstützung im Marketing für Unternehmen als auch Unternehmer:in (Influencer:in).